In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup


def fetch_booking_html(client_id, venue_id, date_yyyymmdd, page=0):
    url = f"https://www.tennisvenues.com.au/booking/{client_id}/fetch-booking-data"

    payload = {
        "client_id": client_id,
        "venue_id": venue_id,
        "resource_id": "",
        "date": date_yyyymmdd,
        "page": page
    }

    response = requests.get(url, params=payload, verify=False)

    if response.status_code != 200:
        raise Exception(f"Error HTTP {response.status_code} para {client_id}")

    return response.text


def parse_booking_table(ajax_html):
    soup = BeautifulSoup(ajax_html, "html.parser")
    booking_tables = soup.find_all("table", class_="BookingSheet")

    if not booking_tables:
        raise Exception("No encontré la tabla BookingSheet")

    booking_table = booking_tables[0]
    rows = booking_table.find_all("tr")

    courts = []
    data = []

    # encabezado
    header_cells = rows[0].find_all(["th", "td"])
    for cell in header_cells[1:]:
        courts.append(cell.get_text(" ", strip=True))

    # filas de horarios
    for row in rows[1:]:
        cells = row.find_all(["th", "td"])

        if len(cells) < 2:
            continue

        time_text = cells[0].get_text(" ", strip=True)

        # si la celda del horario está vacía, la saltamos
        if not time_text:
            continue

        for i, cell in enumerate(cells[1:]):
            court_name = courts[i] if i < len(courts) else f"Court_{i+1}"
            cell_text = cell.get_text(" ", strip=True)
            cell_classes = cell.get("class", [])

            if "NotAvailable" in cell_classes:
                status = "not_available"
            elif "Available" in cell_classes:
                status = "available"
            else:
                status = "unknown"

            data.append({
                "time": time_text,
                "court": court_name,
                "status": status,
                "text": cell_text,
                "classes": ", ".join(cell_classes)
            })

    return pd.DataFrame(data)


def get_booking_dataframe(client_id, venue_id, date_yyyymmdd, page=0):
    ajax_html = fetch_booking_html(
        client_id=client_id,
        venue_id=venue_id,
        date_yyyymmdd=date_yyyymmdd,
        page=page
    )
    return parse_booking_table(ajax_html)


def get_available_courts_for_time(client_id, venue_id, date_yyyymmdd, selected_time, page=0):
    df = get_booking_dataframe(
        client_id=client_id,
        venue_id=venue_id,
        date_yyyymmdd=date_yyyymmdd,
        page=page
    )

    df_time = df[df["time"] == selected_time].copy()
    df_available = df_time[df_time["status"] == "available"].copy()

    return df_available


def get_available_court_names(client_id, venue_id, date_yyyymmdd, selected_time, page=0):
    df_available = get_available_courts_for_time(
        client_id=client_id,
        venue_id=venue_id,
        date_yyyymmdd=date_yyyymmdd,
        selected_time=selected_time,
        page=page
    )

    return df_available["court"].tolist()

In [2]:
from tennisvenues_scraper import (
    get_booking_dataframe,
    get_available_courts_for_time,
    get_available_court_names
)

ModuleNotFoundError: No module named 'tennisvenues_scraper'